# Trabajo Práctico #4
## Problemas de Difusión
### Univesidad de la Marina Mercante
### Facultad de Ingeniería - Ingeniería en Sistemas

- **Profesor:** Dr. Alejandro Gronskis
- **Alumno:** Daniel Ise
- **Legajo:** 28547
- **Fecha:** Noviembre, 2026

## Condiciones de borde

El código **(A)** actualiza los valores para el rango `[1:-1]`, esto es, para todos los valores, salvo el primero y el último. Como el valor de la variable se fija en la frontera, estamos ante una condición de borde de **Dirichlet**.

El código de la función en **(B)** actualiza los valores interiores, `[1:-1]`, para luego tomar los valores en `[1]` y `[-2]` para actualizar los valores extremos, que corresponde a una condición de borde de **Neumann**.

El código **(C)** computa el valor en `[0]` con el valor del último nodo `[-1]`, *conectando* los extremos del *arreglo*, constituyendo por lo tanto una condición de borde **Periódica**.

## Trabajo de Código: reacción - difusión

Primeramente planteamos los parámetros iniciales:

In [ ]:
import numpy
from matplotlib import pyplot, cm

Lx, Ly = 5.0, 5.0
nx, ny = 192, 192
dx, dy = Lx / (nx - 1), Ly / (ny - 1)

Du, Dv = 0.00016, 0.00008
F, k = 0.035, 0.065

t = 8000.0
dt = 9.0 * dx**2 / (40.0 * max(Du, Dv))
nt = int(t / dt)

Planteamos la condición inicial `u0 = 1` y `v0 = 0`:

In [ ]:
numpy.random.seed(42)
num_blocks = 30
randx = numpy.random.randint(1, nx - 1, num_blocks)
randy = numpy.random.randint(1, ny - 1, num_blocks)

u0 = numpy.ones((nx, ny))
v0 = numpy.zeros((nx, ny))
r = 10

for i, j in zip(randx, randy):
    u0[i - r:i + r, j - r:j + r] = 0.50
    v0[i - r:i + r, j - r:j + r] = 0.25

u0 += 0.05 * numpy.random.random((nx, ny))
v0 += 0.05 * numpy.random.random((nx, ny))

Simulamos con condición de borde de Neumann:

In [ ]:
u = u0.copy()
v = v0.copy()

for n in range(nt):
    un = u.copy()
    vn = v.copy()
    
    # laplaciano centrado en espacio
    lap_u = (un[2:, 1:-1] + un[:-2, 1:-1] + un[1:-1, 2:] + un[1:-1, :-2] - 4 * un[1:-1, 1:-1]) / dx**2
    lap_v = (vn[2:, 1:-1] + vn[:-2, 1:-1] + vn[1:-1, 2:] + vn[1:-1, :-2] - 4 * vn[1:-1, 1:-1]) / dx**2
    
    # reacción
    uvv = un[1:-1, 1:-1] * vn[1:-1, 1:-1]**2
    
    # actualizacion temporal adelantando en tiempo
    u[1:-1, 1:-1] = un[1:-1, 1:-1] + dt * (Du * lap_u - uvv + F * (1.0 - un[1:-1, 1:-1]))
    v[1:-1, 1:-1] = vn[1:-1, 1:-1] + dt * (Dv * lap_v + uvv - (F + k) * vn[1:-1, 1:-1])
    
    # condición de borde de Neumann homogénea
    u[0, :] = u[1, :]
    u[-1, :] = u[-2, :]
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]
    
    v[0, :] = v[1, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = v[:, 1]
    v[:, -1] = v[:, -2]

Y, finalmente, graficamos con la función provista:

In [ ]:
fig, ax = pyplot.subplots(ncols=2, figsize=(9.0, 4.0))
ax[0].imshow(u, cmap=cm.RdBu)
ax[0].set_title('Especie U (t = 8000s)')
ax[0].axis('off')

ax[1].imshow(v, cmap=cm.RdBu)
ax[1].set_title('Especie V (t = 8000s)')
ax[1].axis('off')
pyplot.show()